# Notebook 39 — Probe-detected Grokking in DPO (Retrospective)

**Hypothesis (H1)**: During DPO training, probe AUROC for 'model produces preference-shifted output' undergoes a phase transition (sharp step) BEFORE greedy decoding output diverges from baseline. If true, this is a probe-based macroscopic observable for grokking in preference learning — filling exactly the gap that Information-Theoretic Progress Measures (Aug 2024) identifies for circuit-level approaches.

**Null (H0)**: probe scores and behavior change move gradually together — no phase transition, no grokking.

**Method (retrospective on nb37 checkpoints)**:
1. Load nb37 DPO checkpoints (steps 0/20/40/60/80 from `dpo_run/`)
2. For each: generate on 20 hold-out prompts, capture L31 residual at end-of-think, score with FabricationGuard + ReasonGuard probes
3. Plot probe scores vs step, compute fresh-probe AUROC base-vs-final on intermediate activations
4. Phase transition signature: sharp jump at some step → grokking; smooth ramp → no grokking

**Drive**: `/content/drive/MyDrive/openinterp_runs/39_grokking_retrospective/`

**Compute**: ~5 checkpoints × 20 prompts × ~60s/gen = **~100 minutes on RTX 6000**.

**Falsifiable outcome**: regardless of result, ship honestly. Phase transition → novel finding for paper. Gradual → 'preference learning is non-grokking-like in this regime', also publishable as honest negative.


## Phase 1 — Drive mount + checkpoint dir


In [ ]:
from pathlib import Path
import os, sys, time, json

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print(f'Drive mount FAILED: {e}'); raise

DRIVE_ROOT = Path('/content/drive/MyDrive')
assert DRIVE_ROOT.exists()
NB_NAME = '39_grokking_retrospective'
OUT = DRIVE_ROOT / 'openinterp_runs' / NB_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / '_dry_run.txt').write_text('drive mount OK')

# nb37 checkpoint location
NB37_OUT = DRIVE_ROOT / 'openinterp_runs' / '37_multiprobe_dpo_full'
DPO_RUN = NB37_OUT / 'dpo_run'
print(f'✓ nb39 OUT: {OUT}')
print(f'✓ nb37 dpo_run: {DPO_RUN}')
print(f'  exists: {DPO_RUN.exists()}')
if DPO_RUN.exists():
    print(f'  contents: {sorted(p.name for p in DPO_RUN.iterdir())[:20]}')


## Phase 1.5 — Deps


In [ ]:
!pip install -q -U torchao
!pip install -q -U transformers accelerate datasets
!pip install -q -U peft huggingface_hub scikit-learn
print('✓ deps ok')


## Phase 2 — Qwen3.6-27B base load + HF login


In [ ]:
import torch, numpy as np
from huggingface_hub import login, hf_hub_download, HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer

CFG = {
    'model_id':         'Qwen/Qwen3.6-27B',
    'capture_layer_fg': 31,
    'capture_layer_rg': 55,
    'n_holdout':        20,
    'max_new_tokens':   1024,
    'random_seed':      42,
    'fg_probe_repo':    'caiovicentino1/FabricationGuard-linearprobe-qwen36-27b',
    'rg_probe_repo':    'caiovicentino1/ReasoningGuard-linearprobe-qwen36-27b',
    'output_repo':      'caiovicentino1/openinterp-39-grokking-retrospective',
}
THINK_CLOSE_ID = 248069
torch.manual_seed(CFG['random_seed']); np.random.seed(CFG['random_seed'])

HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN is None:
    import getpass; HF_TOKEN = getpass.getpass('HF token: ')
login(HF_TOKEN, add_to_git_credential=False)

device = 'cuda'; assert torch.cuda.is_available()
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✓ {torch.cuda.get_device_name(0)}, {gpu_mem_gb:.1f} GB')

tok = AutoTokenizer.from_pretrained(CFG['model_id'])
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_id'], torch_dtype=torch.bfloat16, device_map='auto',
)
model.eval()
print(f'✓ Base model loaded. Layers: {len(model.model.layers)}')


## Phase 3 — Discover checkpoints + LoRA auto-fix

Inherits the auto-fix from nb37 Phase 7 (POC-style key prefix bug). Checkpoints from save_steps=20 should exist at `dpo_run/checkpoint-{N}/adapter_model.safetensors`.


In [ ]:
from safetensors.torch import load_file, save_file
from peft import PeftModel, LoraConfig, get_peft_model

# Discover available checkpoints
checkpoints = []
if DPO_RUN.exists():
    for d in sorted(DPO_RUN.iterdir()):
        if d.is_dir() and d.name.startswith('checkpoint-'):
            adapter = d / 'adapter_model.safetensors'
            if adapter.exists():
                step = int(d.name.split('-')[1])
                checkpoints.append({'step': step, 'path': d, 'adapter': adapter})
checkpoints.sort(key=lambda x: x['step'])
print(f'Found {len(checkpoints)} checkpoints:')
for c in checkpoints:
    print(f"  step {c['step']}: {c['adapter']} ({c['adapter'].stat().st_size / 1e6:.1f} MB)")

# Also include final saved LoRA
FINAL_LORA = NB37_OUT / 'lora_final'
if (FINAL_LORA / 'adapter_model.safetensors').exists():
    checkpoints.append({'step': 80, 'path': FINAL_LORA, 'adapter': FINAL_LORA / 'adapter_model.safetensors', 'is_final': True})
    print(f'  step 80 (lora_final): {FINAL_LORA}')

if not checkpoints:
    print('⚠️ NO CHECKPOINTS FOUND — nb37 may have cleaned them up')
    print('   Will only test base vs final lora')


In [ ]:
# Auto-fix function (from nb37 Phase 7)
def fix_lora_keys(adapter_path, out_path):
    state = load_file(str(adapter_path))
    keys = list(state.keys())
    if not keys:
        return adapter_path  # empty
    sample = keys[0]
    double_prefix = sample.count('base_model.model.') >= 2
    no_default = '.default.' not in sample and '.lora_' in sample
    if not (double_prefix or no_default):
        return adapter_path  # already fine
    fixed = {}
    for k, v in state.items():
        new_k = k
        if no_default:
            new_k = new_k.replace('.lora_A.weight', '.lora_A.default.weight')
            new_k = new_k.replace('.lora_B.weight', '.lora_B.default.weight')
        fixed[new_k] = v
    out_path.parent.mkdir(parents=True, exist_ok=True)
    save_file(fixed, str(out_path))
    return out_path

def load_lora_into_model(base_model, adapter_dir, fixed_adapter_path):
    """Apply LoRA from a checkpoint dir to the base model. Returns PeftModel."""
    # First time: build PEFT structure with same config as nb37
    if not isinstance(base_model, PeftModel):
        # Reconstruct LoRA config from adapter_config.json
        config_path = adapter_dir / 'adapter_config.json'
        if config_path.exists():
            cfg = json.loads(config_path.read_text())
            lora_cfg = LoraConfig(
                r=cfg.get('r', 16),
                lora_alpha=cfg.get('lora_alpha', 32),
                target_modules=cfg.get('target_modules', None),
                lora_dropout=cfg.get('lora_dropout', 0.0),
                bias=cfg.get('bias', 'none'),
                task_type='CAUSAL_LM',
            )
        else:
            lora_cfg = LoraConfig(r=16, lora_alpha=32, task_type='CAUSAL_LM')
        peft_model = get_peft_model(base_model, lora_cfg)
    else:
        peft_model = base_model
    # Load weights
    state = load_file(str(fixed_adapter_path))
    msg = peft_model.load_state_dict(state, strict=False)
    return peft_model, msg

# Test fix on first checkpoint
if checkpoints:
    test_path = OUT / 'fixed_adapters' / 'checkpoint-test.safetensors'
    fixed = fix_lora_keys(checkpoints[0]['adapter'], test_path)
    state = load_file(str(fixed))
    sample_key = list(state.keys())[0]
    print(f'Sample fixed key: {sample_key}')


## Phase 4 — Load FabricationGuard + ReasonGuard probes


In [ ]:
import joblib

# Download probes from HF
fg_path = hf_hub_download(repo_id=CFG['fg_probe_repo'], filename='probe.joblib', repo_type='dataset')
rg_path = hf_hub_download(repo_id=CFG['rg_probe_repo'], filename='probe.joblib', repo_type='dataset')
fg_probe = joblib.load(fg_path)
rg_probe = joblib.load(rg_path)
print(f'✓ FG probe: {type(fg_probe).__name__}')
print(f'✓ RG probe: {type(rg_probe).__name__}')

def fg_score(activation):
    """Returns P(fabrication) ∈ [0,1]."""
    x = activation.float().cpu().numpy().reshape(1, -1)
    return float(fg_probe.predict_proba(x)[0, 1])

def rg_score(activation):
    """Returns P(unfaithful reasoning) ∈ [0,1]."""
    x = activation.float().cpu().numpy().reshape(1, -1)
    return float(rg_probe.predict_proba(x)[0, 1])


## Phase 5 — Hooks at L31/L55 + hold-out prompts from nb37


In [ ]:
captured = {}
_pos = {'pos': None}

def make_hook(layer_idx):
    def hook(module, input, output):
        h = output[0] if isinstance(output, tuple) else output
        pos = _pos['pos']
        if pos is None or pos >= h.shape[1]:
            return
        captured[f'L{layer_idx}'] = h[0, pos, :].detach().cpu().to(torch.float16).clone()
    return hook

# Register hooks on the underlying base model layers (works whether peft-wrapped or not)
def get_base_layers(m):
    if hasattr(m, 'base_model'):
        return m.base_model.model.model.layers
    return m.model.layers

hook_handles = []
for L in [CFG['capture_layer_fg'], CFG['capture_layer_rg']]:
    handle = get_base_layers(model)[L].register_forward_hook(make_hook(L))
    hook_handles.append(handle)
print(f'✓ Hooks at L{CFG["capture_layer_fg"]}, L{CFG["capture_layer_rg"]}')

# Load nb37 pairs.json for hold-out prompts
pairs_path = NB37_OUT / 'pairs.json'
with open(pairs_path) as f:
    all_pairs = json.load(f)
print(f'Loaded {len(all_pairs)} pairs from nb37')

# Reproducible hold-out
rng = np.random.default_rng(CFG['random_seed'])
indices = rng.choice(len(all_pairs), size=CFG['n_holdout'], replace=False)
holdout = [all_pairs[i] for i in indices]
print(f'Hold-out: {len(holdout)} prompts')
print(f'Source dist: {dict(zip(*np.unique([p["src"] for p in holdout], return_counts=True)))}')


## Phase 6 — Generate per checkpoint + score

For each checkpoint (including base = step 0), generate on hold-out prompts and capture activations.


In [ ]:
from tqdm.auto import tqdm

def find_end_think(token_ids):
    ids = token_ids.tolist() if hasattr(token_ids, 'tolist') else list(token_ids)
    for i in range(len(ids) - 1, -1, -1):
        if ids[i] == THINK_CLOSE_ID:
            return i
    return None

def gen_and_score(prompt):
    messages = [{'role': 'user', 'content': prompt}]
    prompt_text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    enc = tok(prompt_text, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    attention_mask = enc.get('attention_mask', torch.ones_like(input_ids)).to(device)
    n_input = input_ids.shape[1]
    with torch.no_grad():
        gen = model.generate(input_ids, attention_mask=attention_mask,
                             max_new_tokens=CFG['max_new_tokens'],
                             do_sample=False, pad_token_id=tok.eos_token_id)
    full_ids = gen[0]
    output_ids = full_ids[n_input:]
    end_pos = find_end_think(full_ids)
    output_text = tok.decode(output_ids, skip_special_tokens=False)
    if '</think>' in output_text:
        cot = output_text.split('</think>', 1)[0].strip()
        answer = output_text.split('</think>', 1)[1].strip()
    else:
        cot, answer = output_text.strip(), ''
    if end_pos is not None:
        captured.clear()
        _pos['pos'] = end_pos
        with torch.no_grad():
            _ = model(full_ids.unsqueeze(0).to(device))
        fg = fg_score(captured.get(f'L{CFG["capture_layer_fg"]}'))
        rg = rg_score(captured.get(f'L{CFG["capture_layer_rg"]}'))
        act_fg = captured.get(f'L{CFG["capture_layer_fg"]}').clone()
    else:
        fg = rg = float('nan')
        act_fg = None
    return {'cot_len': len(cot), 'answer_len': len(answer), 'fg': fg, 'rg': rg, 'act_fg': act_fg, 'end_pos': end_pos}

# Build checkpoint sequence — base FIRST
ckpt_sequence = [{'step': 0, 'path': None, 'adapter': None, 'name': 'base'}] + checkpoints
print(f'Will run {len(ckpt_sequence)} checkpoints × {CFG["n_holdout"]} prompts = {len(ckpt_sequence) * CFG["n_holdout"]} generations')


In [ ]:
import gc

results_path = OUT / 'per_checkpoint_results.jsonl'
acts_dir = OUT / 'acts'
acts_dir.mkdir(exist_ok=True)

# Resume support
done_keys = set()
if results_path.exists():
    with open(results_path) as f:
        for line in f:
            try:
                r = json.loads(line)
                done_keys.add(f"step{r['step']}_{r['pair_id']}")
            except: continue
    print(f'Resume: {len(done_keys)} (step, pair) combos already done')

for ck_i, ck in enumerate(ckpt_sequence):
    print(f'\n=== Checkpoint {ck_i+1}/{len(ckpt_sequence)}: step={ck["step"]} ===')
    # Apply LoRA (or remove)
    if ck['step'] == 0:
        # Base model — ensure no adapter
        if isinstance(model, PeftModel):
            print('  Detaching adapter for base...')
            # Unload — simpler: re-load base. But too expensive. Use disable.
            model.disable_adapter_layers()
        active_model_label = 'base (no adapter)'
    else:
        # Apply this checkpoint
        fixed = OUT / 'fixed_adapters' / f"checkpoint-{ck['step']}.safetensors"
        fix_lora_keys(ck['adapter'], fixed)
        # First time: wrap in PEFT; subsequent times: just reload weights
        if not isinstance(model, PeftModel):
            adapter_dir = ck['path']
            cfg_path = adapter_dir / 'adapter_config.json'
            if cfg_path.exists():
                cfg = json.loads(cfg_path.read_text())
                lora_cfg = LoraConfig(
                    r=cfg.get('r', 16), lora_alpha=cfg.get('lora_alpha', 32),
                    target_modules=cfg.get('target_modules', None),
                    task_type='CAUSAL_LM',
                )
            else:
                lora_cfg = LoraConfig(r=16, lora_alpha=32, task_type='CAUSAL_LM')
            model = get_peft_model(model, lora_cfg)
            # Re-register hooks on new structure
            for h in hook_handles:
                try: h.remove()
                except: pass
            hook_handles = []
            for L in [CFG['capture_layer_fg'], CFG['capture_layer_rg']]:
                handle = get_base_layers(model)[L].register_forward_hook(make_hook(L))
                hook_handles.append(handle)
        # Reload state
        state = load_file(str(fixed))
        msg = model.load_state_dict(state, strict=False)
        if isinstance(model, PeftModel):
            model.enable_adapter_layers()
        active_model_label = f'lora step {ck["step"]}'
    print(f'  Active model: {active_model_label}')
    
    # Run generations
    for p in tqdm(holdout, desc=f'step{ck["step"]}'):
        key = f"step{ck['step']}_{p['id']}"
        if key in done_keys:
            continue
        try:
            res = gen_and_score(p['prompt'])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            print(f'OOM on {key}, skipping')
            continue
        # Save activation tensor
        if res['act_fg'] is not None:
            torch.save(res['act_fg'], acts_dir / f"{key}.pt")
        record = {
            'step': ck['step'], 'pair_id': p['id'], 'src': p['src'],
            'cot_len': res['cot_len'], 'answer_len': res['answer_len'],
            'fg': res['fg'], 'rg': res['rg'], 'end_pos': res['end_pos'],
            'has_act': res['act_fg'] is not None,
        }
        with open(results_path, 'a') as f:
            f.write(json.dumps(record) + '\n')
        done_keys.add(key)
    torch.cuda.empty_cache(); gc.collect()

print('\n✓ Phase 6 complete')
(OUT / '_phase6_done.txt').write_text(f'ts={time.time()}')


## Phase 7 — Analysis: phase transition or gradual?

1. **Probe scores per step** — plot mean fg, mean rg, combined per checkpoint. Look for sharp jumps.
2. **Fresh-probe AUROC** — train binary classifier on (base activations, step-80 activations), test on intermediate steps. If AUROC at step 40 < AUROC at step 60 sharply → phase transition.
3. **Behavior change** — fraction of generations where chosen-style output emerges (proxy: fg score below baseline).


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

with open(results_path) as f:
    records = [json.loads(line) for line in f]
df = pd.DataFrame(records)
print(f'Total records: {len(df)}')
print(df.groupby('step')[['fg', 'rg']].agg(['mean', 'std']).round(4))


In [ ]:
# Plot 1: probe scores vs step
import matplotlib.pyplot as plt

agg = df.groupby('step').agg({'fg': ['mean', 'std'], 'rg': ['mean', 'std']}).reset_index()
steps = agg['step'].values
fg_mean = agg[('fg', 'mean')].values
fg_std = agg[('fg', 'std')].values
rg_mean = agg[('rg', 'mean')].values
rg_std = agg[('rg', 'std')].values

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].errorbar(steps, fg_mean, yerr=fg_std, marker='o', capsize=4, label='FabricationGuard')
axes[0].set_xlabel('DPO step'); axes[0].set_ylabel('Mean probe score')
axes[0].set_title('Probe scores vs training step')
axes[0].axhline(fg_mean[0], color='gray', linestyle='--', alpha=0.5, label='base (step 0)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].errorbar(steps, rg_mean, yerr=rg_std, marker='s', color='C1', capsize=4, label='ReasonGuard')
axes[1].set_xlabel('DPO step'); axes[1].set_ylabel('Mean probe score')
axes[1].set_title('ReasonGuard vs training step')
axes[1].axhline(rg_mean[0], color='gray', linestyle='--', alpha=0.5)
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / 'fig_probe_vs_step.png', dpi=150)
plt.show()
print('✓ saved fig_probe_vs_step.png')


In [ ]:
# Test 2: Fresh-probe AUROC base-vs-final, test on intermediates
# Load activations
def load_acts(step_filter):
    acts, ids = [], []
    for r in records:
        if r['step'] == step_filter and r['has_act']:
            key = f"step{r['step']}_{r['pair_id']}"
            p = acts_dir / f'{key}.pt'
            if p.exists():
                acts.append(torch.load(p).float().numpy())
                ids.append(r['pair_id'])
    return np.array(acts), ids

X_base, ids_base = load_acts(0)
X_final, ids_final = load_acts(80) if (df['step'] == 80).any() else (None, None)

if X_base is None or X_final is None or len(X_base) < 5 or len(X_final) < 5:
    print('Insufficient data for fresh-probe analysis')
    fresh_results = {}
else:
    X = np.vstack([X_base, X_final])
    y = np.concatenate([np.zeros(len(X_base)), np.ones(len(X_final))])
    clf = LogisticRegression(C=1.0, max_iter=2000)
    clf.fit(X, y)
    print(f'Fresh probe trained on {len(X_base)} base + {len(X_final)} final activations')
    
    # Test on intermediate steps
    fresh_results = {}
    for step in sorted(df['step'].unique()):
        X_step, _ = load_acts(step)
        if X_step is None or len(X_step) < 3:
            continue
        prob = clf.predict_proba(X_step)[:, 1].mean()
        fresh_results[int(step)] = float(prob)
    print('Fresh probe P(final) per step:')
    for s, p in fresh_results.items():
        print(f'  step {s}: P(final-like) = {p:.3f}')


In [ ]:
# Plot 2: fresh probe trajectory
if fresh_results:
    plt.figure(figsize=(7, 4.5))
    steps_sorted = sorted(fresh_results.keys())
    probs = [fresh_results[s] for s in steps_sorted]
    plt.plot(steps_sorted, probs, marker='D', color='C2', markersize=10)
    plt.axhline(0.5, color='gray', linestyle=':', label='chance')
    plt.xlabel('DPO step'); plt.ylabel('Fresh-probe P(final-like)')
    plt.title('Fresh probe progression — phase transition or gradual?')
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT / 'fig_fresh_probe.png', dpi=150)
    plt.show()

# Phase transition detection: max delta between consecutive steps
if len(fresh_results) >= 3:
    sorted_keys = sorted(fresh_results.keys())
    deltas = [fresh_results[sorted_keys[i+1]] - fresh_results[sorted_keys[i]] for i in range(len(sorted_keys)-1)]
    max_delta = max(deltas)
    max_delta_step = sorted_keys[deltas.index(max_delta) + 1]
    avg_delta = sum(deltas) / len(deltas)
    print(f'Max delta between consecutive steps: {max_delta:.3f} (transitioning at step {max_delta_step})')
    print(f'Avg delta: {avg_delta:.3f}')
    ratio = max_delta / max(avg_delta, 1e-6)
    print(f'Max/avg ratio: {ratio:.2f}x')
    print(f'Heuristic: ratio > 2.0 suggests phase transition; < 1.5 suggests gradual')
    if ratio > 2.0:
        print('  🔴 PHASE TRANSITION SIGNAL — grokking-like')
    elif ratio < 1.5:
        print('  🟢 GRADUAL — no grokking signature')
    else:
        print('  🟡 AMBIGUOUS — needs more data')


## Phase 8 — FINAL_VERDICT + HF push


In [ ]:
verdict = {
    'experiment': 'nb39 grokking retrospective',
    'n_checkpoints': int(df['step'].nunique()),
    'n_holdout': CFG['n_holdout'],
    'probe_scores_per_step': {
        str(int(s)): {
            'fg_mean': float(df[df['step']==s]['fg'].mean()),
            'fg_std': float(df[df['step']==s]['fg'].std()),
            'rg_mean': float(df[df['step']==s]['rg'].mean()),
            'rg_std': float(df[df['step']==s]['rg'].std()),
            'n': int((df['step']==s).sum()),
        } for s in sorted(df['step'].unique())
    },
    'fresh_probe_per_step': {str(s): v for s, v in (fresh_results or {}).items()},
}
if fresh_results and len(fresh_results) >= 3:
    verdict['phase_transition_ratio'] = float(ratio)
    verdict['grokking_signal'] = ratio > 2.0
(OUT / 'FINAL_VERDICT.json').write_text(json.dumps(verdict, indent=2))
print(json.dumps(verdict, indent=2))


In [ ]:
# HF push
from huggingface_hub import HfApi, create_repo
api = HfApi()
try:
    create_repo(CFG['output_repo'], repo_type='dataset', private=False, exist_ok=True, token=HF_TOKEN)
except Exception as e:
    print(e)

# README
readme = f'''---
license: apache-2.0
tags: [grokking, dpo, probing, mechanistic-interpretability]
---

# nb39 — Grokking retrospective on nb37 DPO checkpoints

Tests whether DPO on Qwen3.6-27B (nb37) shows phase-transition learning detectable via probes.

## Hypothesis
Probe AUROC for preference-shifted output undergoes phase transition during DPO training, before greedy decoding diverges.

## Result
See FINAL_VERDICT.json. Grokking signal: {{verdict.get("grokking_signal", "undetermined")}}.
Phase transition ratio: {{verdict.get("phase_transition_ratio", "N/A"):.2f}}.

## Methodology lineage
- Nanda et al. 2023 — Progress measures for grokking via MI
- Information-Theoretic Progress Measures (Aug 2024)
- Lei & Xu 2025 — construct-then-compress
- Anthropic Persona Vectors / Goodfire RLFR — probe-based intervention
'''
(OUT / 'README.md').write_text(readme)

api.upload_folder(folder_path=str(OUT), repo_id=CFG['output_repo'],
                  repo_type='dataset', token=HF_TOKEN,
                  commit_message='nb39 grokking retrospective complete',
                  allow_patterns=['README.md', 'FINAL_VERDICT.json', 'fig_*.png', 'per_checkpoint_results.jsonl'])
print(f'✓ pushed to https://huggingface.co/datasets/{CFG["output_repo"]}')


## Done

Read `FINAL_VERDICT.json`. Three outcomes:

**🔴 Phase transition signal** (max/avg ratio > 2.0) → run extended training nb37-v2 with save_steps=10, more steps, finer granularity. Paper-3 material: 'Grokking signature in DPO via probe-based progress measures'.

**🟢 Gradual** (ratio < 1.5) → preference learning is non-grokking-like in this regime. Honest negative, blog post material. Pivot CoTGuard direction.

**🟡 Ambiguous** → expand experiment with more checkpoints + more hold-out.
